# 🌍 Build Your First AI Agent on AWS
## A Hands-On Demo with Amazon Bedrock AgentCore

**AI Festival Summit — June 2026**

---

### What We're Building

A **Smart Travel Concierge Agent** that can:
- ✈️ Search flights between cities
- 🌤️ Check weather at your destination
- 📊 Generate price comparison charts using Code Interpreter
- 🧠 Remember your preferences across conversations
- 🚀 Deploy to production with one command

### Architecture

```
┌─────────────────────────────────────────────────────┐
│                Amazon Bedrock AgentCore              │
│  ┌───────────┐  ┌──────────┐  ┌─────────────────┐  │
│  │  Runtime   │  │  Memory  │  │  Observability  │  │
│  └─────┬─────┘  └────┬─────┘  └────────┬────────┘  │
│        │              │                  │           │
│  ┌─────┴──────────────┴──────────────────┴───────┐  │
│  │          Strands Agent (Claude Sonnet)         │  │
│  └─────┬──────────────┬──────────────────┬───────┘  │
│        │              │                  │           │
│  ┌─────┴─────┐ ┌─────┴──────┐ ┌────────┴────────┐  │
│  │  Gateway   │ │  Gateway   │ │ Code Interpreter│  │
│  │  (Flights) │ │  (Weather) │ │   (Built-in)    │  │
│  └─────┬─────┘ └─────┬──────┘ └─────────────────┘  │
└────────┼──────────────┼─────────────────────────────┘
         │              │
   ┌─────┴─────┐ ┌─────┴──────┐
   │  Lambda:   │ │  Lambda:   │
   │  Flight    │ │  Weather   │
   │  Search    │ │  Lookup    │
   └───────────┘ └────────────┘
```

---
## 🎬 Act 1: Pre-Flight Check

Let's verify our pre-deployed resources are ready.

In [ ]:
# Verify AWS credentials and region
!aws sts get-caller-identity --region us-west-2

In [ ]:
# Check our pre-deployed Lambda functions
!aws lambda invoke --function-name travel-flight-search \
    --payload '{"origin": "Lagos", "destination": "Cape Town", "date": "2026-07-01"}' \
    --region us-west-2 \
    /tmp/flight-response.json && cat /tmp/flight-response.json | python3 -m json.tool

In [ ]:
# Test the weather Lambda
!aws lambda invoke --function-name travel-weather-lookup \
    --payload '{"city": "Cape Town"}' \
    --region us-west-2 \
    /tmp/weather-response.json && cat /tmp/weather-response.json | python3 -m json.tool

✅ Both Lambda functions are live and returning data. These are the **tools** our agent will use.

> **💡 Key Point for the audience:** In a real scenario, these could be your existing APIs, databases, or any service. AgentCore doesn't care what's behind the tool — it just needs a way to call it.

---
## 🎬 Act 2: Scaffold & First Agent

Let's create our agent project using the **AgentCore CLI**.

> 📌 **Switch to Terminal** to show the interactive wizard

In [ ]:
# Step 1: Install the AgentCore CLI (if not already installed)
!npm install -g @aws/agentcore

In [ ]:
# Step 2: Verify CLI installation
!agentcore --version

### Creating the Project

> 📌 **Run this in your terminal** (the wizard is interactive):
> ```bash
> cd /tmp
> agentcore create
> ```
> 
> Choose:
> - **Project name:** `travel-concierge`
> - **Language:** Python
> - **Framework:** Strands Agents
> - **Model:** Claude Sonnet (via Amazon Bedrock)

For this demo, we have the agent code pre-written. Let's look at it:

In [ ]:
# Let's look at our agent code
!cat ../agent/agent.py

### Key things to notice in the agent code:

1. **Framework-agnostic** — We're using Strands Agents, but you could use LangGraph, CrewAI, or any framework
2. **Model-agnostic** — We're using Claude Sonnet via Bedrock, but you could use any LLM
3. **Tools are just Python functions** — decorated with `@tool`, they become capabilities the agent can use
4. **The system prompt** defines the agent's personality and behavior

> 💡 **This is the entire agent.** No complex infrastructure code, no server setup, no auth boilerplate.

### Local Development

> 📌 **Run in terminal:**
> ```bash
> cd /tmp/travel-concierge
> agentcore dev
> ```
> 
> Then test with: *"Plan me a weekend in Johannesburg"*

---
## 🎬 Act 3: Add Tools — Give the Agent Superpowers

Now let's connect our Lambda functions as **tools** the agent can use.

### What is AgentCore Gateway?

Gateway converts your existing APIs, Lambda functions, and services into **MCP-compatible tools** that any agent can use. Think of it as a universal adapter.

> 📌 **Show in AWS Console:** Navigate to Amazon Bedrock → AgentCore → Gateway

In [ ]:
# Get our Lambda ARNs
import boto3
import json

lambda_client = boto3.client('lambda', region_name='us-west-2')

flight_fn = lambda_client.get_function(FunctionName='travel-flight-search')
weather_fn = lambda_client.get_function(FunctionName='travel-weather-lookup')

print(f"✈️  Flight Search ARN: {flight_fn['Configuration']['FunctionArn']}")
print(f"🌤️  Weather Lookup ARN: {weather_fn['Configuration']['FunctionArn']}")

### Testing Tools Directly

Before wiring them to the agent, let's test the tools from Python:

In [ ]:
# Test flight search
response = lambda_client.invoke(
    FunctionName='travel-flight-search',
    Payload=json.dumps({
        "origin": "Lagos",
        "destination": "Cape Town",
        "date": "2026-07-01"
    })
)
flights = json.loads(json.loads(response['Payload'].read())['body'])
print(f"Found {flights['results_count']} flights:")
for f in flights['flights']:
    print(f"  {f['airline']:25s} | {f['flight_number']} | ${f['price_usd']:>6} | {f['stops']} stop(s) | {f['duration']}")

In [ ]:
# Test weather lookup
response = lambda_client.invoke(
    FunctionName='travel-weather-lookup',
    Payload=json.dumps({"city": "Cape Town"})
)
weather = json.loads(json.loads(response['Payload'].read())['body'])
print(f"🌤️ Weather in {weather['city']}:")
print(f"   Current: {weather['current']['temp_c']}°C, {weather['current']['description']}")
print(f"   Humidity: {weather['current']['humidity']}%")
print(f"\n   5-Day Forecast:")
for day in weather['forecast']:
    print(f"   {day['date']:12s} | {day['min_temp_c']}°C - {day['max_temp_c']}°C | {day['description']} | Rain: {day['chance_of_rain']}%")

### Wiring Tools to the Agent

In our `agent.py`, the tools are defined as simple Python functions that call these Lambdas. The agent decides **when** and **how** to use them based on the user's request.

> 📌 **Show in Console:** AgentCore Gateway configuration — how Lambda functions become MCP tools

### Adding Code Interpreter

AgentCore includes a **built-in Code Interpreter** tool. This lets the agent write and execute Python code on the fly — perfect for:
- Budget calculations
- Price comparison charts
- Data analysis

> 📌 **Run in terminal:**
> ```bash
> agentcore add code-interpreter
> ```

### 🧪 The Big Test: Multi-Tool Orchestration

> 📌 **Ask the agent (in terminal or invoke):**
> 
> *"Find me flights from Lagos to Cape Town for July 1st under $800. Also check the weather in Cape Town for that week. Then create a price comparison chart of the flights you found."*

Watch what happens:
1. Agent calls **flight search** tool → gets flight data
2. Agent calls **weather lookup** tool → gets forecast
3. Agent uses **Code Interpreter** → generates a bar chart
4. Agent synthesizes everything into a helpful response

> 💡 **This is the magic of agentic AI** — the agent autonomously decides which tools to use, in what order, and how to combine the results.

---
## 🎬 Act 4: Add Memory — Make It Personal

Right now, every conversation starts fresh. Let's give our agent **memory** so it can remember user preferences across sessions.

### What is AgentCore Memory?

Managed memory infrastructure that stores:
- User preferences
- Conversation history
- Facts the user has shared

No database setup, no schema design — just `agentcore add memory`.

In [ ]:
# Add memory to our agent
# 📌 Run in terminal:
# agentcore add memory
# agentcore deploy

print("Memory capability added! The agent can now remember user preferences.")

### Testing Memory

> 📌 **Conversation 1 — Tell the agent your preferences:**
> 
> *"Remember that I'm vegetarian, I prefer window seats, and my home airport is OR Tambo Johannesburg."*

> 📌 **Conversation 2 — New session, test recall:**
> 
> *"Plan me a trip to Nairobi next month."*

Watch the agent:
- Search flights **from JNB** (remembered home airport)
- Mention **window seat** preference
- Suggest **vegetarian-friendly** restaurants

> 📌 **Show in Console:** Navigate to AgentCore → Memory → show the stored preferences

> 💡 **This is personalization without a database.** AgentCore manages the memory infrastructure for you.

---
## 🎬 Act 5: Deploy to Production

We've been running locally. Let's deploy this agent to **production** — serverless, secure, and scalable.

In [ ]:
# Deploy to AgentCore Runtime
# 📌 Run in terminal:
# agentcore deploy

print("""
🚀 Deploying to Amazon Bedrock AgentCore Runtime...

What's happening behind the scenes:
  ✅ Packaging agent code
  ✅ Creating secure runtime environment
  ✅ Configuring auto-scaling
  ✅ Setting up networking & IAM
  ✅ Registering tools (Gateway, Code Interpreter)
  ✅ Enabling observability (OpenTelemetry)
  ✅ Connecting memory store

All of this — with ONE command.
""")

In [ ]:
# Test the deployed agent
# 📌 Run in terminal:
# agentcore invoke --prompt "What flights are available from Johannesburg to Dubai next week?"

print("Agent is live in production! 🎉")

### Observability — See What the Agent is Thinking

> 📌 **Show in Console:** Navigate to AgentCore → Observability

You can see:
- **Traces** — every step the agent took, every tool it called
- **Latency** — how long each step took
- **Token usage** — how much the LLM processed
- **Tool invocations** — which tools were called and their responses

> 💡 **In production, you need to know what your agent is doing.** AgentCore gives you full visibility with OpenTelemetry-compatible tracing.

---
## 🎬 Act 6: What Else Can AgentCore Do?

We covered a lot in 35 minutes, but there's more:

| Feature | What It Does | Use Case |
|---------|-------------|----------|
| **Identity** | OAuth/OIDC for agents to act on behalf of users | Book flights using the user's airline account |
| **Evaluation** | LLM-as-a-Judge quality scoring | Ensure agent responses are accurate and helpful |
| **Policy** | Cedar-based access control | Restrict which tools an agent can use per user role |
| **Browser Tool** | Built-in web browsing | Research destinations, read reviews |

### Resources

- 📦 **This demo's code:** [github.com/mlungwsr/Smart-Travel-Concierge-Agent](https://github.com/mlungwsr/Smart-Travel-Concierge-Agent)
- 📚 **AgentCore Samples:** [github.com/awslabs/agentcore-samples](https://github.com/awslabs/agentcore-samples)
- 📖 **Documentation:** [docs.aws.amazon.com/bedrock-agentcore](https://docs.aws.amazon.com/bedrock-agentcore/)
- 🛠️ **AgentCore CLI:** [github.com/aws/agentcore-cli](https://github.com/aws/agentcore-cli)
- 🎓 **Workshop:** [Getting Started with AgentCore](https://catalog.us-east-1.prod.workshops.aws/workshops/850fcd5c-fd1f-48d7-932c-ad9babede979/en-US)

---

## 🙏 Thank You!

**Questions?**